<hr>

# ℹ️ DATA EXPLORATION


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

```text
Understand data and take Notes for Data cleaning, Analysis, ML:
    - Valeurs Foncieres (2020-S2 to 2025-S1) 6 large files in the format of .txt
    - arrondissements.csv (paris)
    - communes.csv (outside of paris)
    - Amentities Tool https://overpass-turbo.eu/
    - Rental data in/outside Paris APIs https://www.observatoire-des-loyers.fr/
```

---
<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe

---
## **💰 Property Sales & Price Trends (DVF)**
**`ValeursFoncieres_2020-S2_2025-S1.CSV`**

What it contains:
- Transaction price (€) 
- Property size (m²)
 - Number of rooms 
- Exact address (geocodable) 
- Property type (apartment, house, etc.) 
- Sale date 
- Historical price evolution (5–6 years)

DVF dataset is used for:

- 📈 Price prediction
- 📊 Market trends
- 🤖 ML models

🎯 TARGET VARIABLE = Valeur fonciere (price)

to clearly see:
- 📈 Market growth over 5–6 years
- 🏠 Difference between different types of properties: houses vs apartments vs ... 
- 📉 Market dips (e.g., COVID effects, gaz inflation cuz of Iran/Israel & USA war)

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name                     | Python data type             | Definition             |
|--------------------------------|------------------------------|------------------------|
| **Date mutation**              | `object` to `date`       | Date of the property transaction (sale date). |
| **Nature mutation**            | `object` to `str`            | Type of transaction (e.g., Vente, Echange, Donation). |
| **Valeur fonciere**            | `float64`          | Transaction price of the property in euros (€). |
| **Commune**                    | `object` to `str`            | Official name of the commune (city/town). |
| **Type local**                 | `object` to `str`            | Property type: Appartement, Dépendance, Maison, or Local industriel, commercial ou assimilé. |
| **Identifiant local**          | `float64` to `str`            | Unique identifier of the property unit (can be used to track or deduplicate records). |
| **Surface reelle bati**        | `float64`          | Built surface area of the property in square meters (m²). |
| **Nombre pieces principales**  | `float64` to `int`            | Number of main rooms (excluding kitchens, bathrooms, etc.). |
| **Surface terrain**            | `float64`          | Land surface area in square meters (m²), mainly relevant for houses and land. |
| **Nombre de lots**             | `int64`            | Number of lots involved in the transaction (useful for multi-unit sales). |
| **Voie**                       | `object` to `str`            | Street name of the property. |
| **No voie**                    | `float64` to `str`            | Street number of the property. |
| **B/T/Q**                      | `object` to `str`            | Additional address detail: Bâtiment (building), Type, or Quartier (section/block). |
| **Code postal**                | `float64` to `int`            | French postal code identifying the location. |
| **Departement**                | `str`            | First two digits of the postal code representing the department (administrative region). |
| **Prix_m2**                | `float64`            | First two digits of the postal code representing the department (administrative region). |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ➕ COMBINE - 6 TXT FILES INTO 1 CSV
Drop the rest of the columns

In [ ]:
import pandas as pd

# List of DVF files
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# Columns to keep
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Identifiant local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain",
    "No voie",  # street number
    "B/T/Q", # Bâtiment, Type, Quartier
    "Voie", # street name
    "Code postal"


]

# Output CSV
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# Specify dtype for columns that may have mixed types
dtype_dict = {
    "Voie": str,
    "No voie": str,
    "Code voie": str,
    "B/T/Q": str,
    "Type local": str
}

with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            low_memory=False,  # safer inference
            dtype=dtype_dict
        ):
            # Fix Type local encoding
            chunk["Type local"] = chunk["Type local"].astype(str).str.encode("latin-1", errors="ignore").str.decode("utf-8", errors="ignore")

            # Convert numeric columns
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")

            # Drop rows with missing price
            chunk = chunk.dropna(subset=["Valeur fonciere"])

            # Calculate price per m² where surface is available
            chunk["Prix_m2"] = chunk["Valeur fonciere"] / chunk["Surface reelle bati"]
            chunk.loc[chunk["Surface reelle bati"].isna(), "Prix_m2"] = None  # handle missing surface

            # Extract department from postal code
            chunk["departement"] = chunk["Code postal"].astype(str).str[:2]

            # Write to CSV
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written
            )

            header_written = True
            del chunk  # free memory

print(f"✅ File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ File saved at ../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv


In [9]:
import pandas as pd

# List of DVF files
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# Columns to keep
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Identifiant local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain",
    "No voie",   # street number
    "B/T/Q",     # building / type / quartier
    "Voie",      # street name
    "Code postal"
]

# Output CSV
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# Fix mixed types
dtype_dict = {
    "Voie": str,
    "No voie": str,
    "B/T/Q": str,
    "Type local": str
}

with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            low_memory=False,
            dtype=dtype_dict
        ):
            # -------------------------
            # FIX ENCODING
            # -------------------------
            chunk["Type local"] = (
                chunk["Type local"]
                .astype(str)
                .str.encode("latin-1", errors="ignore")
                .str.decode("utf-8", errors="ignore")
                .str.strip()
            )

            # -------------------------
            # CONVERT NUMERIC
            # -------------------------
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")
            chunk["Surface terrain"] = pd.to_numeric(chunk["Surface terrain"], errors="coerce")

            # Drop rows with missing price
            chunk = chunk.dropna(subset=["Valeur fonciere"])

            # -------------------------
            # PRICE PER M² (SMART FALLBACK)
            # -------------------------
            chunk["surface_used"] = chunk["Surface reelle bati"]

            # If built surface is missing or zero → use terrain
            mask = chunk["surface_used"].isna() | (chunk["surface_used"] == 0)
            chunk.loc[mask, "surface_used"] = chunk.loc[mask, "Surface terrain"]

            # Compute price per m²
            chunk["Prix_m2"] = chunk["Valeur fonciere"] / chunk["surface_used"]

            # Clean invalid values
            chunk.loc[
                chunk["surface_used"].isna() | (chunk["surface_used"] == 0),
                "Prix_m2"
            ] = None

            # Optional: track which surface was used
            chunk["surface_type"] = "bati"
            chunk.loc[mask, "surface_type"] = "terrain"

            # -------------------------
            # EXTRACT DEPARTEMENT
            # -------------------------
            chunk["departement"] = chunk["Code postal"].astype(str).str[:2]

            # -------------------------
            # WRITE TO CSV
            # -------------------------
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written
            )

            header_written = True
            del chunk  # free memory

print(f"✅ File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ File saved at ../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DISPLAY - HEAD

In [10]:
import pandas as pd
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# Display df DataFrame
print("DATA EXPLORATION:")
display(df.head())
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")

C:\Users\sboub\AppData\Local\Temp\ipykernel_4856\1009436813.py:2: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")


MemoryError: Unable to allocate 1.33 GiB for an array with shape (9, 19908349) and data type float64

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DISPLAY - UNIQUE VALUES

In [4]:
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

Data Types & Missing Values of Each Column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 15 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              object 
 1   Nature mutation            object 
 2   Valeur fonciere            float64
 3   No voie                    float64
 4   B/T/Q                      object 
 5   Voie                       object 
 6   Code postal                float64
 7   Commune                    object 
 8   Nombre de lots             int64  
 9   Type local                 object 
 10  Identifiant local          float64
 11  Surface reelle bati        float64
 12  Nombre pieces principales  float64
 13  Surface terrain            float64
 14  Prix_m2                    float64
dtypes: float64(8), int64(1), object(6)
memory usage: 2.2+ GB


None

Column: Date mutation
Unique values (1804): ['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']

Column: Nature mutation
Unique values (6): ['Vente' "Vente en l'Ã©tat futur d'achÃ¨vement"
 'Vente terrain Ã\xa0 bÃ¢tir' 'Echange' 'Adjudication' 'Expropriation']

Column: Valeur fonciere
Unique values (466667): [3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]

Column: No voie
Unique values (9121): [  nan  347. 1084. ... 8562. 8301. 8423.]

Column: B/T/Q
Unique values (41): [nan 'B' 'C' 'A' 'F' 'L' 'T' 'X' 'O' 'Z' 'D' 'G' 'Q' 'Y' 'I' 'E' 'N' 'H'
 '1' 'W' 'V' 'U' 'S' 'R' 'P' 'M' 'K' 'J' '0' '5' '9' '8' 'b' '7' '2' '4'
 '3' '6' '-' '.' '*']

Column: Voie
Unique values (1125220): ['SAINT JULIEN' 'A LA PEROUSE' 'AUX COMMUNS' ... 'SAINT PIERRE AMELOT'
 'SAINT HONORE D EYLAU' 'DES FOSSES SAINT MARCEL']

Column: Code postal
Unique values (5872): [ 1560.  1250.  1310. ... 97380. 97314. 97330.]

Column: Commune
Unique values (31569

In [8]:
print("Unique values in 'Type local':")
df["Type local"].unique().tolist()

Unique values in 'Type local':


[nan,
 'Maison',
 'Dépendance',
 'Appartement',
 'Local industriel. commercial ou assimilé']

---
## **💲 Rental Prices & Yield Data**

**`INTERIM_API_RENTAL.CSV`**

What it contains:
- Median rent (€ / month) 
- Rent per m² 
- Vacancy rate (%) 
- Rent evolution over time 
- Data by arrondissement / commune

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# api for rental prices


---
## **🚇 CSV LIGNES DE TRANSPORT EN COMMUN**

TRANSPORT dataset is used for:
- a
- b
- c


What it contains:
- Metro, RER, tram, bus lines 
- Stops/stations coordinates 
- Line types & frequency (if available)

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# LIGNES DE TRANSPORT EN COMMUN
df_lignes_transport = pd.read_csv("../data/raw/traces-des-lignes-de-transport-en-commun-idfm.csv", sep=";")
print("Lignes de transport en commun:")
display(df_lignes_transport.head())

Lignes de transport en commun:


,ID,Short Name,Long Name,Route Type,Color,Route URL,Shape,id_ilico,OperatorName,NetworkName,ID_Bus_Contrat,url,Type,long_name_first,geo_point_2d
0,IDFM:C00029,502,502,Bus,FBE324,NaN,"{""coordinates"": [[[2.605063, 48.801975], [2.60...",C00029,Keolis Grand Paris Vallée de la Marne,Marne et Brie,9.0,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,5,"48.78188816972777, 2.6480891283837553"
1,IDFM:C01094,57,57,Bus,6E6E00,NaN,"{""coordinates"": [[[2.409755, 48.863434], [2.41...",C01094,RATP,NaN,NaN,https://www.ratp.fr/sites/default/files/lines-...,HORAIRE|PLAN,5,"48.83515885931267, 2.3687425840691674"
2,IDFM:C02220,Soir,Soir Mennecy,Bus,640082,NaN,NaN,C02220,Keolis Val d'Essonne Deux Vallées,Essonne Sud Est,31.0,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,S,NaN
3,IDFM:C00527,6183,6183,Bus,640082,NaN,"{""coordinates"": [[[2.037603, 48.707024], [2.03...",C00527,Keolis Vélizy Vallée de la Bièvre,Vélizy Vallées,27.0,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,6,"48.72894729625489, 2.0909603606050835"
4,IDFM:C02301,GPSO Bus,GPSO Bus,Bus,FF9900,NaN,"{""coordinates"": [[[2.182615, 48.825516], [2.18...",C02301,Origami / Mobicité,Grand Paris Seine Ouest,NaN,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,G,"48.81986189913045, 2.1925887330951395"


---
## **🌍 API MAPPING**

What it contains:
- Latitude & longitude boundaries 
- Arrondissements polygons 
- Commune mapping 
- INSEE codes

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# api for ameneties
